# Plotting Final — Historical State Window

Purpose: choose a series and a historical month/year, then inspect the **12 observations before that month** and every observation recorded during the selected month. The default experiment is **May 1995**.

Two panels are produced: (1) standardized values relative to the mean and sample standard deviation of the 12 observations *before the selected month*; (2) raw values with rolling means of 2 and 4 observations. The standardization baseline is frozen before the target month to avoid using target-month values to define its own reference. Rolling means are trailing, not centered.

Note: this is a descriptive event-window plot, not yet an asset forward test. A monthly series will naturally contribute one observation in the selected month; higher-frequency series may contribute several. Rolling windows count observations, not calendar months.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Experiment controls: change these values for another historical event ----
SERIES_ID = 'CPIAUCSL'   # Any CSV filename under data/cleaned, without .csv
TARGET_YEAR = 1995
TARGET_MONTH = 5         # 1=January, ..., 5=May, ..., 12=December
LOOKBACK_OBSERVATIONS = 12
ROLLING_WINDOWS = (2, 4)

# Locate data regardless of whether Jupyter starts in the repository root or notebook folder.
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p.resolve() for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Start Jupyter from the repository root or notebook folder.')

csv_path = cleaned_dir / f'{SERIES_ID}.csv'
if not csv_path.exists():
    available = sorted(p.stem for p in cleaned_dir.glob('*.csv'))
    raise FileNotFoundError(f'{csv_path} not found. Available series include: {available[:20]}')

df = pd.read_csv(csv_path)
if not {'date', 'value'}.issubset(df.columns):
    raise ValueError(f'{SERIES_ID}.csv must contain date and value columns; found {list(df.columns)}')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
series = (df.dropna(subset=['date', 'value'])
            .sort_values('date')
            .drop_duplicates('date', keep='last')
            .set_index('date')['value'])

if not 1 <= TARGET_MONTH <= 12:
    raise ValueError('TARGET_MONTH must be between 1 and 12.')
month_start = pd.Timestamp(TARGET_YEAR, TARGET_MONTH, 1)
month_end = month_start + pd.offsets.MonthEnd(1)

# Freeze the reference window strictly before the target month.
history = series.loc[series.index < month_start]
baseline = history.tail(LOOKBACK_OBSERVATIONS)
target_month_data = series.loc[(series.index >= month_start) & (series.index <= month_end)]

if len(baseline) < LOOKBACK_OBSERVATIONS:
    raise ValueError(f'Only {len(baseline)} observations exist before {month_start:%B %Y}; need {LOOKBACK_OBSERVATIONS}.')
if target_month_data.empty:
    raise ValueError(f'No observations for {SERIES_ID} in {month_start:%B %Y}.')

reference_mean = baseline.mean()
reference_std = baseline.std(ddof=1)  # sample standard deviation
if not np.isfinite(reference_std) or reference_std == 0:
    raise ValueError('The 12-observation reference standard deviation is zero or undefined; z-scores cannot be computed.')

# Window contains the 12 prior observations plus all observations in the target month.
window = pd.concat([baseline, target_month_data]).loc[lambda s: ~s.index.duplicated(keep='last')].sort_index()
window_z = (window - reference_mean) / reference_std
rolling_2 = window.rolling(window=ROLLING_WINDOWS[0], min_periods=ROLLING_WINDOWS[0]).mean()
rolling_4 = window.rolling(window=ROLLING_WINDOWS[1], min_periods=ROLLING_WINDOWS[1]).mean()

print(f'Series: {SERIES_ID} | Target: {month_start:%B %Y}')
print(f'Data coverage: {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}')
print(f'Baseline observations: {len(baseline)} | Target-month observations: {len(target_month_data)}')
print(f'Baseline mean: {reference_mean:.6g} | Sample std: {reference_std:.6g}')
display(pd.DataFrame({'value': window, 'z_score_vs_pre_month_baseline': window_z,
                      'rolling_mean_2': rolling_2, 'rolling_mean_4': rolling_4}).tail(LOOKBACK_OBSERVATIONS + len(target_month_data)))

## Event-window plots

The vertical line marks the start of the selected month. The standardized chart uses one fixed mean/std estimated from the preceding 12 observations. The raw-value chart overlays trailing rolling means; the 12-observation rolling mean will only appear after 12 values are available.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True, constrained_layout=True)

# Panel 1: standardized values, using only the pre-event baseline.
axes[0].plot(window_z.index, window_z.values, marker='o', markersize=3, linewidth=1.4,
             label='Z-score vs pre-month mean/std')
axes[0].axhline(0, color='black', linewidth=1, label='Pre-month mean')
axes[0].axhline(1, linestyle='--', alpha=0.65, label='+1 std')
axes[0].axhline(-1, linestyle='--', alpha=0.65, label='−1 std')
axes[0].axhline(2, linestyle=':', alpha=0.65, label='+2 std')
axes[0].axhline(-2, linestyle=':', alpha=0.65, label='−2 std')
axes[0].axvline(month_start, color='black', linestyle='-.', alpha=0.8, label='Target month starts')
axes[0].set_title(f'{SERIES_ID} — standardized event window | {month_start:%B %Y}')
axes[0].set_ylabel('Standard deviations from pre-month mean')
axes[0].legend(ncol=3, fontsize=8)
axes[0].grid(True, alpha=0.2)

# Panel 2: observed level and trailing moving averages.
axes[1].plot(window.index, window.values, marker='o', markersize=3, linewidth=1.4, label='Observed value')
axes[1].plot(rolling_2.index, rolling_2.values, linewidth=1.5, label='Rolling mean (2 observations)')
axes[1].plot(rolling_4.index, rolling_4.values, linewidth=1.8, label='Rolling mean (4 observations)')
axes[1].axvline(month_start, color='black', linestyle='-.', alpha=0.8, label='Target month starts')
axes[1].set_title('Observed series with trailing rolling means')
axes[1].set_ylabel('Original units')
axes[1].set_xlabel('Observation date')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.show()

# Explicitly report the target month's observations for event-by-event review.
print('Target-month observations:')
display(target_month_data.rename('value').to_frame())

## Interpretation checklist

- Is the target-month observation unusually far from the pre-month baseline (in standard-deviation units)?
- Was the move already visible in the preceding 12 observations, or did it emerge during the target month?
- Do the 5- and 12-observation rolling means agree, diverge, or cross?
- Is the pattern plausible for the series' native frequency and economic meaning?

Do not call a point a tradable anomaly based on this plot alone. The next step is to define an event timestamp, an asset, a forward horizon, and a no-look-ahead benchmark.